In [ ]:
!pip install rouge-score
!pip install transformers datasets sentencepiece

import re
import unicodedata
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CKPT_DIR     = "mt5_health_qa/checkpoint-556"
VAL_FILE     = "Val.csv"
TRAIN_FILE   = "Train.csv"

QUESTION_COL = "input"
ANSWER_COL   = "output"
SUBSET_COL   = "subset"
ID_COL       = "ID"

MAX_INPUT_LENGTH = 170
MAX_NEW_TOKENS   = 170
BATCH_SIZE       = 32     # inference only — no gradients, so larger batch is fine
NUM_BEAMS        = 5
NO_REPEAT_NGRAM  = 3
LENGTH_PENALTY   = 1.2

SUBSET_TO_LANGUAGE = {
    "Eng_Uga": "english", "Eng_Gha": "english",
    "Eng_Eth": "english", "Eng_Ken": "english",
    "Aka_Gha": "akan",    "Lug_Uga": "luganda",
    "Swa_Ken": "swahili", "Amh_Eth": "amharic",
}
LANGUAGE_PREFIXES = {
    "english": "Answer this health question: ",
    "akan":    "Bua saa safoɔ a ɛfa apɔwmuden ho: ",
    "luganda": "Ddamu ekibuuzo kino ky'obulamu: ",
    "swahili": "Jibu swali hili la afya: ",
    "amharic": "ይህንን የጤና ጥያቄ ይመልሱ: ",
}

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def add_language_prefix(question, subset):
    lang   = SUBSET_TO_LANGUAGE.get(str(subset).strip(), "english")
    prefix = LANGUAGE_PREFIXES.get(lang, LANGUAGE_PREFIXES["english"])
    return prefix + normalize_text(question)

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)
    r1, rl = [], []
    for pred, ref in zip(predictions, references):
        if pred.strip() and ref.strip():
            s = scorer.score(str(ref), str(pred))
            r1.append(s["rouge1"].fmeasure)
            rl.append(s["rougeL"].fmeasure)
    return {
        "rouge1_f1": float(np.mean(r1)) if r1 else 0.0,
        "rougeL_f1": float(np.mean(rl)) if rl else 0.0,
    }

def competition_score(r1, rl):
    return 0.37 * r1 + 0.37 * rl

In [ ]:
# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
print("Loading data...")
val   = pd.read_csv(VAL_FILE)
train = pd.read_csv(TRAIN_FILE)

val = val.sample(frac=0.5, random_state=100).reset_index(drop=True)

val[QUESTION_COL] = val[QUESTION_COL].fillna("").apply(normalize_text)
val[ANSWER_COL]   = val[ANSWER_COL].fillna("").apply(normalize_text)
val["input_text"] = val.apply(lambda r: add_language_prefix(r[QUESTION_COL], r[SUBSET_COL]), axis=1)

print(f"Full val set: {len(val):,} rows")
print(f"Subset distribution:\n{val[SUBSET_COL].value_counts().to_string()}")

In [ ]:
# ─────────────────────────────────────────────
# RETRIEVAL LOOKUP (train only — no val leakage)
# ─────────────────────────────────────────────
print("\nBuilding retrieval lookup from train set...")
train[QUESTION_COL] = train[QUESTION_COL].fillna("").apply(normalize_text)
train[ANSWER_COL]   = train[ANSWER_COL].fillna("").apply(normalize_text)

lookup = {}
for _, row in train.iterrows():
    key = (str(row[QUESTION_COL]).strip().lower(), str(row[SUBSET_COL]).strip())
    if key not in lookup:
        lookup[key] = str(row[ANSWER_COL])

print(f"Lookup size: {len(lookup):,} entries")

# Check how many val questions are in lookup
val_keys     = val.apply(lambda r: (r[QUESTION_COL].strip().lower(), r[SUBSET_COL].strip()), axis=1)
n_retrievable = val_keys.isin(lookup).sum()
print(f"Val questions retrievable from lookup: {n_retrievable} ({100*n_retrievable/len(val):.1f}%)")

In [ ]:
# ─────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────
print(f"\nLoading model from: {CKPT_DIR}")
device    = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(CKPT_DIR).to(device)
model.eval()
model.config.use_cache = True   # safe to enable for inference
print(f"Model loaded on {device} | Parameters: {model.num_parameters()/1e6:.1f}M")

In [ ]:
# ─────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────
predictions = [None] * len(val)
generate_idxs = []

# Retrieval pass
for i, row in val.iterrows():
    key = (row[QUESTION_COL].strip().lower(), row[SUBSET_COL].strip())
    if key in lookup:
        predictions[i] = lookup[key]
    else:
        generate_idxs.append(i)

print(f"\nRetrieval: {len(val) - len(generate_idxs)} answered")
print(f"Generation needed: {len(generate_idxs)}")

# Generation pass
input_texts = [val.loc[i, "input_text"] for i in generate_idxs]

with torch.no_grad():
    for batch_start in tqdm(range(0, len(input_texts), BATCH_SIZE), desc="Generating"):
        batch = input_texts[batch_start : batch_start + BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        ).to(device)

        outputs = model.generate(
            **inputs,
            num_beams            = NUM_BEAMS,
            max_new_tokens       = MAX_NEW_TOKENS,
            no_repeat_ngram_size = NO_REPEAT_NGRAM,
            length_penalty       = LENGTH_PENALTY,
            early_stopping       = True,
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for idx, pred in zip(generate_idxs[batch_start : batch_start + BATCH_SIZE], decoded):
            predictions[idx] = pred.strip()

# Fill any remaining None
predictions = [p if p else "" for p in predictions]

In [ ]:
# ─────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────
references = val[ANSWER_COL].tolist()
overall    = compute_rouge(predictions, references)
r1 = overall["rouge1_f1"]
rl = overall["rougeL_f1"]

print(f"\n{'='*45}")
print(f"FULL VALIDATION SET RESULTS")
print(f"{'='*45}")
print(f"  ROUGE-1 F1:        {r1:.4f}")
print(f"  ROUGE-L F1:        {rl:.4f}")
print(f"  Competition score: {competition_score(r1, rl):.4f}  (excl. LLM judge)")
print(f"{'='*45}")

# Per-subset breakdown
print(f"\nPer-subset breakdown:")
val["prediction"] = predictions
results = []
for subset, grp in val.groupby(SUBSET_COL):
    s = compute_rouge(grp["prediction"].tolist(), grp[ANSWER_COL].tolist())
    s["subset"]   = subset
    s["language"] = SUBSET_TO_LANGUAGE.get(subset, "unknown")
    s["n"]        = len(grp)
    results.append(s)

results_df = pd.DataFrame(results).sort_values("rouge1_f1", ascending=False)
results_df["combined"] = results_df.apply(
    lambda r: competition_score(r["rouge1_f1"], r["rougeL_f1"]), axis=1
)
print(results_df[["subset", "language", "n", "rouge1_f1", "rougeL_f1", "combined"]].to_string(index=False))